In [13]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys
_here = Path.cwd().resolve()
_root = next(p for p in [_here, *_here.parents] if (p / "src").is_dir())
sys.path.insert(0, str(_root / "src"))

import pandas as pd
from regression_modelling.constants import FEATURE_SOURCES
from regression_modelling.data_wrangling import sources, features
from crime_blockgroup_mapping.config import INTERIM_DIR
from crime_blockgroup_mapping.constants import UCR_YEAR

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
# What sources are declared, and how each is fetched
for name, src in FEATURE_SOURCES.items():
    print(f"{name:12} backend={src.backend:4} location={src.location:10} "
          f"key={src.key_col:6} cols={src.feature_cols}")

vacancy      backend=bq   location=vacancy    key=geoid  cols=('vacant_pct',)
liens        backend=bq   location=liens      key=geoid  cols=('clip_liens_pct',)
foreclosures backend=bq   location=foreclosures key=geoid  cols=('clip_foreclosure_pct',)
seven_eleven backend=bq   location=seven_eleven key=geoid  cols=('unq_seven_eleven_clips',)
gas_stations backend=bq   location=gas_stations key=geoid  cols=('unq_gas_station_clips',)
liquor_stores backend=bq   location=liquor_stores key=geoid  cols=('unq_liquor_store_clips',)


In [ ]:
# Run only when the BQ staging tables don't exist yet or upstream data changed.
# This executes the CREATE OR REPLACE DDL — skip if already built. Left commented on purpose.
#for name in ["vacancy", "liens"]:
#    sources.run_bq_build(name)
#    print(f"built {name}")

In [11]:
# Uncached single pull to confirm BQ auth + schema before running the full pipeline
for source in ["gas_stations","liquor_stores","seven_eleven"]:
    df = sources.run_bq_pull(f"{source}")
    print(f"{source} pull: {df.shape}")

gas_stations pull: (241456, 4)
liquor_stores pull: (241456, 4)
seven_eleven pull: (241456, 4)


In [15]:
# Load liens and vacancy parquet files
for name, src in FEATURE_SOURCES.items():
    df = sources.pull_source(src, refresh=False)
    print(f"{name:12} {df.shape}  cols={list(df.columns)}")

vacancy      (241456, 4)  cols=['geoid', 'vacant_pct', 'vacant_addr', 'total_addr']
liens        (241456, 4)  cols=['geoid', 'clip_liens_pct', 'total_clips', 'clip_w_liens']
foreclosures (241456, 4)  cols=['geoid', 'clip_foreclosure_pct', 'total_unq_clips', 'unq_clip_w_foreclosure']
seven_eleven (241456, 4)  cols=['geoid', 'seven_eleven_clip_pct', 'tot_unq_clips', 'unq_seven_eleven_clips']
gas_stations (241456, 4)  cols=['geoid', 'gas_station_clip_pct', 'tot_unq_clips', 'unq_gas_station_clips']
liquor_stores (241456, 4)  cols=['geoid', 'liquor_store_clip_pct', 'tot_unq_clips', 'unq_liquor_store_clips']


In [5]:
# Load demographic features
demo = features.build_demographic_features(refresh=False)
print("demographic:", demo.shape)
demo.head()

demographic: (242335, 8)


,geoid,det_pct,in_household_pct,moved1yr_pct,Division,city_centers_dist,pop_est_5mile,pop_ch_1mile
0,020130001001,73.279352,65.293602,16.042465,9.0,50.0,2066.0,-37.826087
1,020130001002,57.459677,65.293602,16.042465,9.0,50.0,884.0,-11.588785
2,020130001003,75.638051,65.293602,16.042465,9.0,50.0,674.0,-32.747604
3,020160001001,33.962264,66.310160,19.562244,9.0,50.0,1023.0,-6.250000
4,020160002001,9.478673,68.261851,24.745302,9.0,50.0,4411.0,-0.500835


In [16]:
# Assemble features
feats = features.assemble_features(refresh=False)
print("bg_predictors:", feats.shape)
feats.info()

bg_predictors: (242335, 14)
<class 'pandas.DataFrame'>
RangeIndex: 242335 entries, 0 to 242334
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   geoid                   242335 non-null  str    
 1   det_pct                 242335 non-null  float64
 2   in_household_pct        242335 non-null  float64
 3   moved1yr_pct            242335 non-null  float64
 4   Division                242335 non-null  float64
 5   city_centers_dist       239780 non-null  float64
 6   pop_est_5mile           239392 non-null  float64
 7   pop_ch_1mile            239392 non-null  float64
 8   vacant_pct              240235 non-null  float64
 9   clip_liens_pct          241273 non-null  float64
 10  clip_foreclosure_pct    241273 non-null  float64
 11  unq_seven_eleven_clips  241367 non-null  Int64  
 12  unq_gas_station_clips   241367 non-null  Int64  
 13  unq_liquor_store_clips  241367 non-null  Int64  
dtypes: 

In [17]:
# geoid should be a 12-char string, unique per row
print("geoid dtype :", feats["geoid"].dtype)
print("geoid lengths:", feats["geoid"].str.len().value_counts().to_dict())
print("duplicate geoids:", feats["geoid"].duplicated().sum())

# Non-null coverage per column — low coverage on vacancy/liens flags join or fill issues
print("\nNon-null coverage:")
print((feats.notna().mean() * 100).round(1).astype(str) + "%")

geoid dtype : str
geoid lengths: {12: 242335}
duplicate geoids: 0

Non-null coverage:
geoid                     100.0%
det_pct                   100.0%
in_household_pct          100.0%
moved1yr_pct              100.0%
Division                  100.0%
city_centers_dist          98.9%
pop_est_5mile              98.8%
pop_ch_1mile               98.8%
vacant_pct                 99.1%
clip_liens_pct             99.6%
clip_foreclosure_pct       99.6%
unq_seven_eleven_clips     99.6%
unq_gas_station_clips      99.6%
unq_liquor_store_clips     99.6%
dtype: str


In [18]:
print("Cached source pulls:")
for p in sorted((INTERIM_DIR / "sources").glob("*.parquet")):
    print(" ", p.relative_to(INTERIM_DIR.parent), f"{p.stat().st_size/1e6:.1f} MB")

feat_path = INTERIM_DIR / "features" / "bg_predictors.parquet"
print("\nFeature matrix written:", feat_path.exists(), "→", feat_path)

Cached source pulls:
  interim/sources/demographic.parquet 8.0 MB
  interim/sources/foreclosures.parquet 2.7 MB
  interim/sources/gas_stations.parquet 2.7 MB
  interim/sources/liens.parquet 2.6 MB
  interim/sources/liquor_stores.parquet 2.6 MB
  interim/sources/seven_eleven.parquet 2.7 MB
  interim/sources/vacancy.parquet 3.3 MB

Feature matrix written: True → /home/eprashar_solutions_corelogic_com/crime-idx-2026/data/interim/features/bg_predictors.parquet
